# TWINIA Servo Dataset Analysis

This notebook downloads the repository, loads the servo motor datasets, generates plots, and summarizes the experimental files used during the doctoral internship project on the quadruped robot **TWINIA**.

The analysis covers:
- Step response experiments
- Ramp response experiments
- Chirp excitation experiments
- Sim-to-real validation support
- Energy analysis with INA219-based monitoring

## 1. Clone the GitHub repository

Run the following cell to download the project from GitHub.

In [ ]:
!rm -rf twinia-servo-digital-twin
!git clone https://github.com/cordero1794/twinia-servo-digital-twin.git

In [ ]:
%cd twinia-servo-digital-twin

## 2. Install required libraries

In [ ]:
!pip install pandas matplotlib -q

## 3. Verify dataset files

In [ ]:
import os

dataset_dir = 'dataset'
files = sorted(os.listdir(dataset_dir)) if os.path.exists(dataset_dir) else []
print('Dataset directory exists:', os.path.exists(dataset_dir))
print('Files found:')
for f in files:
    print('-', f)

## 4. Load datasets

In [ ]:
import glob
import pandas as pd

csv_files = sorted(glob.glob('dataset/*.csv'))
datasets = {}

for file_path in csv_files:
    name = os.path.basename(file_path)
    df = pd.read_csv(file_path)
    datasets[name] = df

print(f'Total CSV files loaded: {len(datasets)}')
list(datasets.keys())

## 5. Preview one dataset

In [ ]:
sample_name = list(datasets.keys())[0]
print('Previewing:', sample_name)
datasets[sample_name].head()

## 6. Display dataset summary

In [ ]:
summary_rows = []

for name, df in datasets.items():
    summary_rows.append({
        'file': name,
        'rows': len(df),
        'columns': len(df.columns),
        'test_type': df['test_type'].iloc[0] if 'test_type' in df.columns and len(df) > 0 else 'N/A',
        'platforms': ', '.join(sorted(df['platform'].dropna().astype(str).unique())) if 'platform' in df.columns else 'N/A'
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

## 7. Create figures directory

In [ ]:
figures_dir = 'figures'
os.makedirs(figures_dir, exist_ok=True)
print('Figures will be saved in:', figures_dir)

## 8. Plot helper functions

In [ ]:
import matplotlib.pyplot as plt

def plot_variable(df, x_col, y_col, title, output_path):
    plt.figure(figsize=(10, 5))
    plt.plot(df[x_col], df[y_col], linewidth=2)
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.show()
    plt.close()

def plot_combined(df, x_col, y_cols, title, output_path):
    plt.figure(figsize=(12, 6))
    for col in y_cols:
        if col in df.columns:
            plt.plot(df[x_col], df[col], linewidth=2, label=col)
    plt.xlabel(x_col)
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.show()
    plt.close()

## 9. Generate plots for all datasets

In [ ]:
required_columns = ['time_s', 'voltage_V', 'current_A', 'position_deg', 'velocity_deg_s', 'torque_Nm']

for name, df in datasets.items():
    print(f'Processing {name}...')
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        print(f'Skipping {name}. Missing columns: {missing}')
        continue

    base = os.path.splitext(name)[0]

    plot_variable(df, 'time_s', 'voltage_V', f'Voltage vs Time - {base}', f'{figures_dir}/{base}_voltage.png')
    plot_variable(df, 'time_s', 'current_A', f'Current vs Time - {base}', f'{figures_dir}/{base}_current.png')
    plot_variable(df, 'time_s', 'position_deg', f'Position vs Time - {base}', f'{figures_dir}/{base}_position.png')
    plot_variable(df, 'time_s', 'velocity_deg_s', f'Velocity vs Time - {base}', f'{figures_dir}/{base}_velocity.png')
    plot_variable(df, 'time_s', 'torque_Nm', f'Torque vs Time - {base}', f'{figures_dir}/{base}_torque.png')
    plot_combined(
        df,
        'time_s',
        ['voltage_V', 'current_A', 'position_deg', 'velocity_deg_s', 'torque_Nm'],
        f'Combined Variables - {base}',
        f'{figures_dir}/{base}_combined.png'
    )

print('All plots generated successfully.')

## 10. Compare test families

The next cells group files by test type.

In [ ]:
step_files = [k for k in datasets if 'step' in k]
ramp_files = [k for k in datasets if 'ramp' in k]
chirp_files = [k for k in datasets if 'chirp' in k]

print('Step files:', step_files)
print('Ramp files:', ramp_files)
print('Chirp files:', chirp_files)

## 11. Overlay current curves for step tests

In [ ]:
plt.figure(figsize=(10, 5))
for name in step_files:
    df = datasets[name]
    plt.plot(df['time_s'], df['current_A'], linewidth=2, label=name)
plt.xlabel('time_s')
plt.ylabel('current_A')
plt.title('Step Tests - Current Comparison')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(f'{figures_dir}/step_current_comparison.png', dpi=300)
plt.show()
plt.close()

## 12. Overlay current curves for ramp tests

In [ ]:
plt.figure(figsize=(10, 5))
for name in ramp_files:
    df = datasets[name]
    plt.plot(df['time_s'], df['current_A'], linewidth=2, label=name)
plt.xlabel('time_s')
plt.ylabel('current_A')
plt.title('Ramp Tests - Current Comparison')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(f'{figures_dir}/ramp_current_comparison.png', dpi=300)
plt.show()
plt.close()

## 13. Overlay current curves for chirp tests

In [ ]:
plt.figure(figsize=(10, 5))
for name in chirp_files:
    df = datasets[name]
    plt.plot(df['time_s'], df['current_A'], linewidth=2, label=name)
plt.xlabel('time_s')
plt.ylabel('current_A')
plt.title('Chirp Tests - Current Comparison')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(f'{figures_dir}/chirp_current_comparison.png', dpi=300)
plt.show()
plt.close()

## 14. Maximum current summary

In [ ]:
max_current_rows = []

for name, df in datasets.items():
    max_current_rows.append({
        'file': name,
        'max_current_A': df['current_A'].max(),
        'min_voltage_V': df['voltage_V'].min(),
        'max_position_deg': df['position_deg'].max(),
        'max_torque_Nm': df['torque_Nm'].max()
    })

max_summary_df = pd.DataFrame(max_current_rows).sort_values('max_current_A', ascending=False)
max_summary_df

## 15. Save summary tables

In [ ]:
summary_df.to_csv('docs/dataset_summary.csv', index=False)
max_summary_df.to_csv('docs/dataset_max_summary.csv', index=False)
print('Summary tables saved in docs/.')

## 16. Final notes

This notebook supports reproducible analysis of the TWINIA servo datasets. It helps visualize the relationship between current consumption, voltage behavior, actuator motion, and torque estimates during the doctoral internship workflow.